# FrameWeave Ovis2.5-9B ablations — HERBench lite_v2

Notebook **chỉ eval**. Không train. Oracle $K=8$, greedy, parser chữ cái. Output sai vẫn ở mẫu số.

So sánh sáu biến thể trên cùng manifest 1,971 câu / 68 video. Smoke 2 câu ghi `eval_smoke/` và **không** vào bảng paper. Full eval rồi `aggregate`.

## Source code

Mở notebook này từ GitHub hoặc Colab. Run All clone [FrameWeave](https://github.com/nqMinh-GR/FrameWeave.git) vào `/content/FrameWeave`, rồi symlink `ovis/reljoin`, `ablation/`, `configs/` và `run_ablation.py` vào `/content`. Không upload file source.

Suite ablation seed 42 (`token_mlp`, `global_only`, `local_only`, `hybrid`, `hybrid_no_time`) là lần train lại và không được phát hành. Notebook này không lấy các adapter đó từ GitHub hay Drive. Vanilla không cần checkpoint.

## Hardware

Colab **A100 40 GB**. Disk trống ≥ ~55 GB (Ovis2.5-9B ~18 GB + HERBench videos ~35 GB).

`SMOKE_FIRST=True` chạy 2 câu trước. Sau khi smoke ổn, đặt `SMOKE_FIRST=False` rồi chạy full. `RUN_FULL=True` eval 1,971 câu và ghi bảng.


## 1. Configuration


In [ ]:
# 1. Eval-only config — FrameWeave Ovis ablations
from __future__ import annotations
from pathlib import Path

assert Path("/content").exists(), "Notebook này chỉ chạy trên Google Colab (/content)."

ROOT = Path("/content")
SMOKE_FIRST = True
RUN_FULL = True
VARIANT = "all"
SEED = 42

MODEL_ID = "ATH-MaaS/Ovis2.5-9B"
MODEL_REVISION = "5a1f9b995c98ae5b85a28ec89d88d74295be0ea7"
HERBENCH_ID = "DanBenAmi/HERBench"
HERBENCH_REVISION = "30b42e97180f2c7c0fdc3f2aed412ae04f0366ca"
HERBENCH_BASE_SHA256 = "6104884c366906d5c223cc9727c3bad778dcdf98c881c4b17b556c20e940af97"
HERBENCH_ACC_SHA256 = "eb67b618f3004b52c8290b2fd30e4012153a66f3f93d576c29760280caa3344d"
HERBENCH_BASE_URL = "https://raw.githubusercontent.com/DanBenAmi/HERBench/main/evaluation/model_wrappers/base_vlm.py"
HERBENCH_ACC_URL = "https://raw.githubusercontent.com/DanBenAmi/HERBench/main/evaluation/calculate_accuracy.py"

VENV = ROOT / "ovis_env"
PYTHON = VENV / "bin" / "python"
ABLATION = ROOT
OUT_DIR = ABLATION / "outputs" / "frameweave_ovis_ablation"
MODEL_DIR = ROOT / "checkpoints" / "ovis2_5_9b"
HERBENCH_ROOT = ROOT / "HERBench_official"
HERBENCH_VIDEOS = HERBENCH_ROOT / "videos"
EVAL_DIR = ROOT / "herbench_eval_pins"
PERSIST = Path("/content/drive/MyDrive/frameweave_ovis_ablation_outputs")
TRAINABLE = ["token_mlp", "global_only", "local_only", "hybrid", "hybrid_no_time"]

print({
    "SMOKE_FIRST": SMOKE_FIRST,
    "RUN_FULL": RUN_FULL,
    "VARIANT": VARIANT,
    "OUT_DIR": str(OUT_DIR),
    "MODEL_DIR": str(MODEL_DIR),
    "backbone": MODEL_ID,
})


## 2. Environment (`ovis_env`)

Official Ovis2.5-9B pins: `torch==2.4.0`, `transformers==4.51.3`, `flash-attn==2.7.0.post2`.
Stop if flash-attn install fails — the ViT hard-imports it.


In [ ]:
# 2. Bootstrap isolated ovis_env (Python 3.11 + torch 2.4.0 + flash-attn 2.7)
from __future__ import annotations
import os, subprocess
from pathlib import Path

assert Path("/content").exists(), "This notebook targets Google Colab (/content)."
ROOT = Path("/content")
VENV = ROOT / "ovis_env"
VENV_BIN = VENV / "bin"
PYTHON = VENV_BIN / "python"
PIP = VENV_BIN / "pip"
LOG = ROOT / "outputs" / "frameweave_ovis_ablation" / "logs" / "cell4_env.log"
LOG.parent.mkdir(parents=True, exist_ok=True)
LOG.write_text("", encoding="utf-8")
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def sh(cmd, check=True, env=None):
    line = "$ " + " ".join(map(str, cmd))
    print(line, flush=True)
    with LOG.open("a", encoding="utf-8") as f:
        f.write(line + "\n")
        proc = subprocess.run(
            list(map(str, cmd)),
            check=False,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            env=env,
        )
        f.write(proc.stdout or "")
        f.write(f"\nexit={proc.returncode}\n\n")
    print(proc.stdout or "", end="")
    if check and proc.returncode != 0:
        print("LOG:", LOG)
        print((proc.stdout or "")[-4000:])
        raise RuntimeError(f"command failed ({proc.returncode}). See {LOG}")
    return proc

sh(["apt-get", "update", "-qq"], check=False)
sh(["apt-get", "install", "-y", "-qq", "python3.11", "python3.11-venv", "python3.11-dev", "ffmpeg"], check=False)
py311 = Path("/usr/bin/python3.11")
assert py311.exists(), "Không tìm thấy python3.11 — kiểm tra apt install."

need_recreate = True
if PYTHON.exists():
    ver = subprocess.check_output([str(PYTHON), "-V"], text=True).strip()
    print("Existing venv:", ver)
    need_recreate = "3.11" not in ver
if need_recreate:
    if VENV.exists():
        sh(["rm", "-rf", str(VENV)])
    sh([str(py311), "-m", "venv", "--copies", str(VENV)])
if not PIP.exists():
    sh([str(PYTHON), "-m", "ensurepip", "--upgrade"], check=False)
if not PIP.exists():
    get_pip = Path("/tmp/get-pip.py")
    sh(["curl", "-sSL", "https://bootstrap.pypa.io/get-pip.py", "-o", str(get_pip)])
    sh([str(PYTHON), str(get_pip)])
sh([str(PYTHON), "-m", "pip", "install", "-q", "--upgrade", "pip", "setuptools<70", "wheel"])
print("Python in venv:", subprocess.check_output([str(PYTHON), "-V"], text=True).strip())
sh([
    str(PYTHON), "-m", "pip", "install", "--disable-pip-version-check",
    "torch==2.4.0", "torchvision==0.19.0",
    "--index-url", "https://download.pytorch.org/whl/cu121",
])
sh([
    str(PYTHON), "-m", "pip", "install", "--disable-pip-version-check", "--upgrade-strategy", "only-if-needed",
    "transformers==4.51.3", "numpy==1.25.0", "pillow==10.3.0", "moviepy==1.0.3",
    "huggingface_hub", "hf-transfer", "datasets", "accelerate", "einops", "sentencepiece",
    "opencv-python-headless", "pandas", "scipy", "tqdm", "ninja", "packaging",
])
print("Installing flash-attn==2.7.0.post2 (required by Ovis ViT)...")
fa_env = os.environ.copy()
fa_env["MAX_JOBS"] = "4"
sh(
    [str(PYTHON), "-m", "pip", "install", "flash-attn==2.7.0.post2", "--no-build-isolation"],
    env=fa_env,
)
check = subprocess.check_output([
    str(PYTHON), "-c",
    "import torch, transformers, flash_attn; "
    "print(torch.__version__); print(transformers.__version__); print(flash_attn.__version__); "
    "print(torch.__file__); print(transformers.__file__)"
], text=True)
print(check)
for line in check.splitlines():
    if "/site-packages/" in line:
        assert "/content/ovis_env/" in line, f"Package leaked outside venv: {line}"
print("ovis_env ready:", PYTHON)
print("LOG:", LOG)


## 3. Drive + pin HERBench evaluators


In [ ]:
# 3. Drive + pin HERBench evaluator (skip download if SHA matches)
from __future__ import annotations

import hashlib
import os
import time
import urllib.request
from pathlib import Path

ROOT = Path("/content")
HERBENCH_ROOT = ROOT / "HERBench_official"
HERBENCH_VIDEOS = HERBENCH_ROOT / "videos"
MODEL_DIR = ROOT / "checkpoints" / "ovis2_5_9b"
OUT_DIR = ROOT / "outputs" / "frameweave_ovis_ablation"
EVAL_DIR = ROOT / "herbench_eval_pins"
HERBENCH_BASE_URL = "https://raw.githubusercontent.com/DanBenAmi/HERBench/main/evaluation/model_wrappers/base_vlm.py"
HERBENCH_ACC_URL = "https://raw.githubusercontent.com/DanBenAmi/HERBench/main/evaluation/calculate_accuracy.py"
HERBENCH_BASE_SHA256 = "6104884c366906d5c223cc9727c3bad778dcdf98c881c4b17b556c20e940af97"
HERBENCH_ACC_SHA256 = "eb67b618f3004b52c8290b2fd30e4012153a66f3f93d576c29760280caa3344d"
JSDELIVR_BASE = "https://cdn.jsdelivr.net/gh/DanBenAmi/HERBench@main/evaluation/model_wrappers/base_vlm.py"
JSDELIVR_ACC = "https://cdn.jsdelivr.net/gh/DanBenAmi/HERBench@main/evaluation/calculate_accuracy.py"

for path in [HERBENCH_ROOT, HERBENCH_VIDEOS, MODEL_DIR.parent, EVAL_DIR, OUT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PERSIST = Path("/content/drive/MyDrive/frameweave_ovis_ablation_outputs")
except Exception as exc:
    print("Drive mount skipped:", exc)
    PERSIST = OUT_DIR / "persist_local"
if PERSIST.resolve() == OUT_DIR.resolve():
    PERSIST = OUT_DIR / "persist_local"
    print("Persist dir was out_dir; using", PERSIST)
PERSIST.mkdir(parents=True, exist_ok=True)
os.environ["RELJOIN_PERSIST"] = str(PERSIST)
os.environ["RELJOIN_ROOT"] = str(ROOT)
os.environ["OVIS_ABLATION_ROOT"] = str(ROOT)


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def fetch_pinned(urls: list[str], dest: Path, expected: str) -> None:
    if dest.exists() and sha256_file(dest) == expected:
        print("Already pinned:", dest.name)
        return
    last_err = None
    for url in urls:
        for attempt in range(4):
            try:
                print(f"GET {url} (try {attempt + 1})")
                urllib.request.urlretrieve(url, dest)
                got = sha256_file(dest)
                if got == expected:
                    print("OK", dest.name)
                    return
                last_err = RuntimeError(f"SHA mismatch for {dest.name}: {got}")
            except Exception as exc:
                last_err = exc
                time.sleep(2 ** attempt)
    raise RuntimeError(f"Cannot pin {dest.name}: {last_err}")


fetch_pinned([HERBENCH_BASE_URL, JSDELIVR_BASE], EVAL_DIR / "base_vlm.py", HERBENCH_BASE_SHA256)
fetch_pinned([HERBENCH_ACC_URL, JSDELIVR_ACC], EVAL_DIR / "calculate_accuracy.py", HERBENCH_ACC_SHA256)
print({"PERSIST": str(PERSIST), "OUT_DIR": str(OUT_DIR), "EVAL_DIR": str(EVAL_DIR)})


## 4. Clone source and verify


In [ ]:
# Fetch FrameWeave source. Run All clones the repo; do not upload .py files.
from __future__ import annotations

import subprocess
from pathlib import Path

REPO_URL = "https://github.com/nqMinh-GR/FrameWeave.git"
REPO_DIR = Path("/content/FrameWeave")
ROOT = Path("/content")
assert ROOT.exists(), "This notebook targets Google Colab (/content)."

if not (REPO_DIR / ".git").exists():
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)])
else:
    subprocess.check_call(["git", "-C", str(REPO_DIR), "pull", "--ff-only"])


def link(src: Path, dest: Path) -> None:
    src = src.resolve()
    if dest.is_symlink():
        if dest.resolve() != src:
            dest.unlink()
            dest.symlink_to(src)
        return
    if dest.exists():
        raise RuntimeError(
            f"{dest} already exists and is not the repo link. Restart the runtime, then Run All."
        )
    dest.symlink_to(src)


link(REPO_DIR / "ovis" / "reljoin", ROOT / "reljoin")
link(REPO_DIR / "ovis_ablation" / "ablation", ROOT / "ablation")
link(REPO_DIR / "ovis_ablation" / "configs", ROOT / "configs")
link(REPO_DIR / "ovis_ablation" / "run_ablation.py", ROOT / "run_ablation.py")
print("FrameWeave source linked from", REPO_DIR / "ovis_ablation", "and", REPO_DIR / "ovis" / "reljoin")


In [ ]:
# 4. Verify ablation sources (eval needs the runner and the V6 package)
from __future__ import annotations
from pathlib import Path

ROOT = Path("/content")
required = [
    ROOT / "run_ablation.py",
    ROOT / "ablation" / "eval_herbench.py",
    ROOT / "ablation" / "aggregate.py",
    ROOT / "reljoin" / "reljoin_modules.py",
    ROOT / "reljoin" / "herbench_oracle.py",
    ROOT / "reljoin" / "ovis_generate.py",
    ROOT / "herbench_eval_pins" / "base_vlm.py",
    ROOT / "herbench_eval_pins" / "calculate_accuracy.py",
]
missing = [str(path) for path in required if not path.is_file()]
assert not missing, "Missing files — clone or symlink failed:\n" + "\n".join(missing)
print("sources OK", len(required))


## 5. Gắn checkpoint `best/adapter.pt`

Suite seed 42 không có trong repo và không được kéo từ Drive. Vanilla không cần file.


In [ ]:
# 5. Seed-42 ablation suite is a retrain and is not published.
# Do not fetch token_mlp, global_only, local_only, hybrid, or hybrid_no_time from GitHub or Drive.
from __future__ import annotations
from pathlib import Path

ROOT = Path("/content")
try:
    SEED
except NameError:
    SEED = 42
try:
    VARIANT
except NameError:
    VARIANT = "all"
OUT = ROOT / "outputs" / "frameweave_ovis_ablation"
TRAINABLE = ["token_mlp", "global_only", "local_only", "hybrid", "hybrid_no_time"]
if VARIANT.strip().lower() == "all":
    names = list(TRAINABLE)
else:
    names = [part.strip() for part in VARIANT.split(",") if part.strip() and part.strip() != "vanilla"]

missing = []
for name in names:
    local = OUT / f"seed_{SEED}" / name / "best" / "adapter.pt"
    if not local.is_file():
        missing.append(str(local))
if missing:
    raise FileNotFoundError(
        "Suite ablation seed 42 là lần train lại và không được phát hành. "
        "Notebook này không lấy các adapter đó từ GitHub hay Drive.\n" + "\n".join(missing)
    )
print("checkpoints OK", names or ["vanilla"])


## 6. Tải Ovis2.5-9B + HERBench lite videos (~35 GB)

PerceptionTest không tải — không train. Cell này bắt buộc trước smoke (`model_config` phải `True`).


In [ ]:
# 6. Download Ovis2.5-9B + HERBench lite (eval only)
from __future__ import annotations

import os
import shutil
import subprocess
from pathlib import Path

ROOT = Path("/content")
PYTHON = ROOT / "ovis_env" / "bin" / "python"
assert PYTHON.exists(), "Chạy cell 2 trước"
os.environ["PATH"] = f"{ROOT / 'ovis_env' / 'bin'}:{os.environ.get('PATH', '')}"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

HERBENCH_ROOT = ROOT / "HERBench_official"
HERBENCH_VIDEOS = HERBENCH_ROOT / "videos"
MODEL_DIR = ROOT / "checkpoints" / "ovis2_5_9b"
HERBENCH_HF_ID = "DanBenAmi/HERBench"
HERBENCH_REVISION = "30b42e97180f2c7c0fdc3f2aed412ae04f0366ca"
MODEL_HF_ID = "ATH-MaaS/Ovis2.5-9B"
MODEL_REVISION = "5a1f9b995c98ae5b85a28ec89d88d74295be0ea7"

for path in [HERBENCH_ROOT, HERBENCH_VIDEOS, MODEL_DIR.parent]:
    path.mkdir(parents=True, exist_ok=True)

print("Downloading Ovis2.5-9B (~18 GB bf16)...")
subprocess.run([str(PYTHON), "-c", f"""
import os
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
from huggingface_hub import snapshot_download
snapshot_download(
    repo_id='{MODEL_HF_ID}',
    revision='{MODEL_REVISION}',
    local_dir=r'{MODEL_DIR}',
    ignore_patterns=['*.msgpack', '*.h5', 'flax_model*', 'tf_model*'],
)
print('model OK')
"""], check=True)
assert (MODEL_DIR / "config.json").exists(), f"Missing {MODEL_DIR / 'config.json'}"

print("Downloading HERBench lite (~35GB)...")
subprocess.run([str(PYTHON), "-c", f"""
import os
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
from huggingface_hub import snapshot_download
snapshot_download(
    repo_id='{HERBENCH_HF_ID}',
    repo_type='dataset',
    revision='{HERBENCH_REVISION}',
    local_dir=r'{HERBENCH_ROOT}',
    allow_patterns=[
        'data/herbench_lite_v2.parquet',
        'data/herbench_annotations_lite_v2.json',
        'data/*metadata.json',
        'videos/videos.tar.part.00',
        'videos/videos.tar.part.01',
        'videos/videos.tar.part.02',
        'videos/videos.tar.part.03',
        'videos/videos_lite_info.txt',
        'videos/videos.tar.checksums.txt',
    ],
)
print('herbench snapshot OK')
"""], check=True)

LITE_MP4_MIN = 68
parts = sorted(HERBENCH_VIDEOS.glob("videos.tar.part.*"))
print("parts:", [path.name for path in parts])
mp4s = list(HERBENCH_VIDEOS.rglob("*.mp4"))
assert len(parts) >= 4 or len(mp4s) >= 2, (
    f"Thiếu videos.tar.part.00–03 trong {HERBENCH_VIDEOS}. Có: {[path.name for path in parts]}"
)
if len(mp4s) < LITE_MP4_MIN:
    assert len(parts) >= 4, (
        f"Chỉ có {len(mp4s)} mp4 và thiếu tar parts — tải lại videos.tar.part.00–03."
    )
    cat = subprocess.Popen(["cat", *map(str, parts)], stdout=subprocess.PIPE)
    subprocess.run(["tar", "-xf", "-", "-C", str(HERBENCH_VIDEOS)], stdin=cat.stdout, check=True)
    cat.wait()
    print("Extracted archives")
else:
    print(f"Already have {len(mp4s)} mp4")

n_mp4 = len(list(HERBENCH_VIDEOS.rglob("*.mp4")))
if n_mp4 >= LITE_MP4_MIN:
    for part in parts:
        part.unlink(missing_ok=True)
    print("Removed archives after full extract")
else:
    print(f"KEEP tar parts — only {n_mp4} mp4 (need ~{LITE_MP4_MIN} for full eval)")

print({
    "videos_mp4": n_mp4,
    "model_config": (MODEL_DIR / "config.json").exists(),
    "free_GiB": round(shutil.disk_usage(ROOT).free / 2**30, 2),
})
assert (MODEL_DIR / "config.json").exists(), "Ovis config.json missing — tải lại cell 6."
assert n_mp4 >= 2, "Lite extract xong vẫn thiếu mp4"
if n_mp4 < LITE_MP4_MIN:
    print(f"WARN: {n_mp4} mp4 < {LITE_MP4_MIN}. Smoke có thể chạy; full eval sẽ thiếu video.")
print("Downloads OK")


## 7. Smoke (2 câu) rồi eval full

Smoke ghi `eval_smoke/` và không đụng `eval/seed_*/predictions.jsonl`. Full eval xong mới `aggregate`. `Complete=no` chưa phải lite_v2 đầy đủ.


In [ ]:
# 7. HERBench oracle eval — smoke then full (NO training)
from __future__ import annotations
import json
import subprocess
from pathlib import Path

ROOT = Path("/content")
PYTHON = ROOT / "ovis_env" / "bin" / "python"
ABLATION = ROOT
OUT = ABLATION / "outputs" / "frameweave_ovis_ablation"
assert PYTHON.exists(), "Chạy cell 2 trước"

try:
    SMOKE_FIRST
except NameError:
    SMOKE_FIRST = True
try:
    RUN_FULL
except NameError:
    RUN_FULL = True
try:
    VARIANT
except NameError:
    VARIANT = "all"
try:
    SEED
except NameError:
    SEED = 42

def run(args: list[str]) -> None:
    cmd = [str(PYTHON), "run_ablation.py", *args]
    print("$", " ".join(cmd), flush=True)
    subprocess.run(cmd, cwd=str(ABLATION), check=True)

if SMOKE_FIRST:
    run(["eval", "--variant", VARIANT, "--seed", str(SEED), "--limit", "2"])
    metrics_path = OUT / "eval_smoke" / f"seed_{SEED}" / "metrics.json"
    metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
    smoke_predictions = OUT / "eval_smoke" / f"seed_{SEED}" / "predictions.jsonl"
    assert metrics["paper_result"] is False and metrics["smoke"] is True
    assert smoke_predictions.is_file(), smoke_predictions
    print("Smoke metrics:", {key: metrics[key] for key in ("paper_result", "limit", "n_selected", "n_predictions")})
    print("Smoke is not a paper result. Predictions:", smoke_predictions)

if RUN_FULL:
    run(["eval", "--variant", VARIANT, "--seed", str(SEED)])
    run(["aggregate"])
    table = (OUT / "aggregate" / "results.md").read_text(encoding="utf-8")
    print(table)
    if "| no |" in table:
        print("Aggregate rows with Complete=no are not a full lite_v2 result.")


## 8. Bảng Overall / QA / họ task


In [ ]:
# 8. Show aggregate tables. Complete=no is not a paper number.
from __future__ import annotations
from pathlib import Path

OUT = ROOT / "outputs" / "frameweave_ovis_ablation" / "aggregate"
for name in ("results.md", "results.csv", "results.tex"):
    path = OUT / name
    print(path, "exists" if path.is_file() else "MISSING")
md_path = OUT / "results.md"
if md_path.is_file():
    text = md_path.read_text(encoding="utf-8")
    print(text)
    if "| no |" in text:
        print("Complete=no: bảng này chưa phải kết quả lite_v2 đầy đủ.")
else:
    print("Chưa có aggregate. Chạy full eval ở cell 7.")


## 9. Copy artifact sang Drive


In [ ]:
# 9. Persist eval, eval_smoke, and aggregate to Drive + zip
from __future__ import annotations
import shutil
from datetime import datetime, timezone
from pathlib import Path

ROOT = Path("/content")
OUT = ROOT / "outputs" / "frameweave_ovis_ablation"
persist = Path("/content/drive/MyDrive/frameweave_ovis_ablation_outputs")
if not persist.parent.exists():
    persist = OUT / "persist_local"
persist.mkdir(parents=True, exist_ok=True)
for part in ("eval", "eval_smoke", "aggregate", "shared"):
    src = OUT / part
    if not src.exists():
        print("skip missing", src)
        continue
    dest = persist / part
    if persist.resolve() == OUT.resolve():
        continue
    shutil.copytree(src, dest, dirs_exist_ok=True)
    print("Copied", src, "→", dest)

stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
zip_path = ROOT / f"frameweave_ovis_ablation_eval_{stamp}"
if OUT.exists():
    shutil.make_archive(str(zip_path), "zip", root_dir=OUT)
    print("Zip:", f"{zip_path}.zip")
print("Persist:", persist)
